# Query published FTW polygons for an AOI

This notebook mirrors `examples/17_query_published_ftw_polygons.py`.

It demonstrates two workflows:

1. A fully local synthetic FTW-like GeoParquet tile store and manifest. This is deterministic and safe for CI-style validation.
2. An optional live query against the public Fields of The World (FTW) GeoParquet source using `agribound.query_ftw(...)` with no `manifest_path` or `tile_dir`.

The live query retrieves already-published FTW prediction polygons. It does not run FTW inference, use Google Earth Engine, or treat FTW as ground truth.


## Setup

Run from a development checkout of AgriBound on the `query-ftw-polygons` branch.

If needed:

```bash
pip install -e ".[dev,docs]"
```

The public online backend uses PyArrow Dataset/S3 support through the existing `pyarrow` dependency.


In [ ]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

import agribound as ab

print("agribound:", ab.__version__)

## Build a tiny local FTW-like tile store

This mirrors the runnable `.py` example. It creates:

- two GeoParquet tiles;
- a manifest with `out_path` and `status` fields;
- a small cross-tile AOI.

This is not real FTW data. It is a small deterministic fixture for validating query behavior.


In [ ]:
def build_synthetic_ftw_store(base_dir: Path) -> tuple[Path, Path, Path]:
    """Create a tiny FTW-like tile store for a runnable local example."""
    tile_dir = base_dir / "ftw_tiles"
    tile_dir.mkdir(parents=True, exist_ok=True)

    tile_west = gpd.GeoDataFrame(
        {
            "field_id": ["west-1", "west-2", "old-west"],
            "geometry_hash": ["hash-west-1", "hash-west-2", "hash-old-west"],
            "label": ["field", "field", "field"],
            "time": ["2025-01-01", "2025-01-01", "2024-01-01"],
        },
        geometry=[
            box(-100.80, 40.10, -100.25, 40.65),
            box(-100.45, 40.55, -100.05, 40.90),
            box(-100.70, 40.20, -100.35, 40.45),
        ],
        crs="EPSG:4326",
    )
    tile_west_path = tile_dir / "tile_west.parquet"
    tile_west.to_parquet(tile_west_path, index=False)

    tile_east = gpd.GeoDataFrame(
        {
            "field_id": ["east-1", "east-2", "west-1"],
            "geometry_hash": ["hash-east-1", "hash-east-2", "hash-west-1-dup"],
            "label": ["field", "non_field_background", "field"],
            "time": ["2025-01-01", "2025-01-01", "2025-01-01"],
        },
        geometry=[
            box(-99.90, 40.15, -99.20, 40.80),
            box(-99.70, 40.25, -99.35, 40.55),
            box(-100.80, 40.10, -100.25, 40.65),
        ],
        crs="EPSG:4326",
    )
    tile_east_path = tile_dir / "tile_east.parquet"
    tile_east.to_parquet(tile_east_path, index=False)

    manifest = gpd.GeoDataFrame(
        {
            "tile_id": ["tile_west", "tile_east"],
            "out_path": [tile_west_path.name, tile_east_path.name],
            "status": ["ok", "ok"],
        },
        geometry=[
            box(-101.0, 40.0, -100.0, 41.0),
            box(-100.0, 40.0, -99.0, 41.0),
        ],
        crs="EPSG:4326",
    )
    manifest_path = base_dir / "ftw_tile_manifest.parquet"
    manifest.to_parquet(manifest_path, index=False)

    aoi_path = base_dir / "demo_aoi.geojson"
    aoi = gpd.GeoDataFrame(
        {"name": ["cross_tile_aoi"]},
        geometry=[box(-100.55, 40.30, -99.55, 40.75)],
        crs="EPSG:4326",
    )
    aoi.to_file(aoi_path, driver="GeoJSON")

    return manifest_path, tile_dir, aoi_path


workspace = Path(tempfile.mkdtemp(prefix="agribound-ftw-query-"))
manifest_path, tile_dir, aoi_path = build_synthetic_ftw_store(workspace)

print("workspace:", workspace)
print("manifest_path:", manifest_path)
print("tile_dir:", tile_dir)
print("aoi_path:", aoi_path)

## Query local synthetic tiles

This validates the local manifest/tile workflow retained by the PR.


In [ ]:
clipped = ab.query_ftw(
    study_area=aoi_path,
    year=2025,
    label="field",
    clip=True,
    source_backend="manifest",
    manifest_path=manifest_path,
    tile_dir=tile_dir,
    output_path=workspace / "ftw_demo_clipped.parquet",
)

print(f"Clipped AOI result: {len(clipped)} polygons")
clipped[["field_id", "source_tile_id", "label", "time", "geometry"]]

In [ ]:
full_polygons = ab.query_ftw(
    study_area=[-100.55, 40.30, -99.55, 40.75],
    year=2025,
    label="field",
    clip=False,
    source_backend="manifest",
    manifest_path=manifest_path,
    tile_dir=tile_dir,
)

empty = ab.query_ftw(
    study_area=[-95.0, 35.0, -94.0, 36.0],
    year=2025,
    label="field",
    source_backend="manifest",
    manifest_path=manifest_path,
    tile_dir=tile_dir,
)

summary = {
    "clipped_count": int(len(clipped)),
    "unclipped_count": int(len(full_polygons)),
    "empty_count": int(len(empty)),
}
print(json.dumps(summary, indent=2))

assert len(clipped) == 3
assert len(full_polygons) == 3
assert len(empty) == 0

In [ ]:
aoi = gpd.read_file(aoi_path)

ax = clipped.plot(figsize=(7, 5), edgecolor="black", alpha=0.5)
aoi.boundary.plot(ax=ax, linewidth=2)
ax.set_title("Synthetic FTW-like query result")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

## Live public FTW query

This cell uses the intended default behavior:

```python
ab.query_ftw(study_area=..., year=2025)
```

No `manifest_path` or `tile_dir` is provided. The function queries the public FTW GeoParquet source using the PyArrow backend.

The AOI below is intentionally small to keep the query lightweight.


In [ ]:
live_aoi = [-93.55, 41.90, -93.50, 41.95]
live_output = workspace / "ftw_live_smoke.parquet"

live = ab.query_ftw(
    study_area=live_aoi,
    year=2025,
    label="field",
    clip=True,
    output_path=live_output,
    max_features=1000,
)

print("count:", len(live))
print("crs:", live.crs)
print("columns:", live.columns.tolist())
live.head()

## Validate the live output file

The written GeoParquet should be readable by GeoPandas.


In [ ]:
loaded = gpd.read_parquet(live_output)

print("read-back count:", len(loaded))
print("read-back crs:", loaded.crs)
loaded.head()

In [ ]:
# Basic sanity checks for the live query.
assert len(live) > 0, "The live AOI query returned no polygons."
assert live.crs is not None
assert loaded.crs is not None
assert len(loaded) == len(live)

if "label" in live.columns:
    assert set(live["label"].dropna().unique()) == {"field"}

if "time" in live.columns:
    years = gpd.pd.to_datetime(live["time"], errors="coerce").dt.year.dropna().unique()
    assert set(years) == {2025}

live_bounds = gpd.GeoDataFrame(geometry=[box(*live_aoi)], crs="EPSG:4326")
assert live.intersects(live_bounds.geometry.iloc[0]).all()

print("Live query sanity checks passed.")

## Plot live query result

The blue boundary is the AOI. The returned FTW polygons are shown underneath.


In [ ]:
live_bounds = gpd.GeoDataFrame(geometry=[box(*live_aoi)], crs="EPSG:4326")

ax = live.plot(figsize=(8, 6), edgecolor="black", alpha=0.45)
live_bounds.boundary.plot(ax=ax, linewidth=2)
ax.set_title("Live public FTW query result")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

## Optional interactive map

This requires `folium`. It is skipped automatically if `folium` is not installed.


In [ ]:
try:
    import folium

    def _folium_safe(gdf):
        """Return a copy with JSON-serializable non-geometry columns."""
        out = gdf.copy()

        for col in out.columns:
            if col == out.geometry.name:
                continue

            if str(out[col].dtype).startswith("datetime64"):
                out[col] = out[col].astype(str)
                continue

            out[col] = out[col].map(
                lambda value: value.isoformat() if hasattr(value, "isoformat") else value
            )

        for col in out.select_dtypes(include=["object"]).columns:
            if col == out.geometry.name:
                continue

            out[col] = out[col].map(
                lambda value: (
                    value if isinstance(value, (str, int, float, bool, type(None))) else str(value)
                )
            )

        return out

    minx, miny, maxx, maxy = live_aoi
    center = [(miny + maxy) / 2, (minx + maxx) / 2]
    m = folium.Map(location=center, zoom_start=13, tiles="OpenStreetMap")

    live_map = _folium_safe(live).to_crs("EPSG:4326")

    folium.GeoJson(
        live_map.to_json(),
        name="FTW polygons",
        style_function=lambda _: {
            "color": "#1f77b4",
            "weight": 1,
            "fillOpacity": 0.25,
        },
    ).add_to(m)

    folium.GeoJson(
        live_bounds.to_json(),
        name="AOI",
        style_function=lambda _: {
            "color": "#d62728",
            "weight": 3,
            "fillOpacity": 0,
        },
    ).add_to(m)

    folium.LayerControl().add_to(m)
    m
except ImportError:
    print("folium is not installed; skipping interactive map.")

## Notes

- Use a small AOI first. The public FTW GeoParquet corpus is large.
- `max_features` is useful for smoke tests and previews.
- For larger AOIs, omit `max_features` only after confirming query performance.
- FTW polygons are model-derived predictions and should be treated as an inferred comparison product, not ground truth.
